In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import math

from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()


x_train = x_train / 255.0
x_test = x_test / 255.0


x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

def build_model(lr=0.01):

    model = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    optimizer = optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model
def step_decay(epoch):
    initial_lr = 0.01
    drop = 0.5
    epochs_drop = 5

    lr = initial_lr * math.pow(drop, epoch // epochs_drop)
    return lr
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)
def train_model(name, cb_list=None, lr=0.01):

    print(f"\nTraining: {name}")

    model = build_model(lr)

    # Track learning rate
    class LRHistory(callbacks.Callback):
        def on_epoch_begin(self, epoch, logs=None):
            if not hasattr(self, 'lrs'):
                self.lrs = []
            self.lrs.append(
                float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
            )

    lr_track = LRHistory()

    callbacks_list = [lr_track]

    if cb_list:
        callbacks_list += cb_list

    history = model.fit(
        x_train, y_train,
        epochs=10,
        validation_data=(x_test, y_test),
        callbacks=callbacks_list,
        verbose=1
    )

    return history.history['val_loss'], lr_track.lrs
step_scheduler = callbacks.LearningRateScheduler(step_decay, verbose=1)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)
results = {

    "Fixed LR":
        train_model("Fixed LR", None, 0.01),

    "Step Decay":
        train_model("Step Decay", [step_scheduler], 0.01),

    "Adaptive LR":
        train_model("Adaptive LR", [reduce_lr, early_stop], 0.01)
}

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training: Fixed LR


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8985 - loss: 0.3424 - val_accuracy: 0.9471 - val_loss: 0.1838
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9288 - loss: 0.2579 - val_accuracy: 0.9531 - val_loss: 0.1689
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9396 - loss: 0.2247 - val_accuracy: 0.9493 - val_loss: 0.1807
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9448 - loss: 0.2061 - val_accuracy: 0.9600 - val_loss: 0.1587
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9470 - loss: 0.2005 - val_accuracy: 0.9557 - val_loss: 0.1789
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9517 - loss: 0.1870 - val_accuracy: 0.9621 - val_loss: 0.1470
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9527 - loss: 0.1863 - val_accuracy: 0.9613 - val_loss: 0.1551
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9565 - loss: 0.1708 - 